# Floquet gates on a 7-site chain q-c-q-c-q-c-q

The physics and the optimisers live in `floquet/chain.py`; this notebook configures, runs and analyses.
Chain: qubits on sites 0, 2, 4, 6 and tunable couplers on sites 1, 3, 5. A periodic train of Gaussian flux
pulses on the couplers transfers two excitations: `1100_0011` = $(q_0,q_2)\to(q_4,q_6)$,
`1010_0101` = $(q_0,q_4)\to(q_2,q_6)$, `1001_0110` = $(q_0,q_6)\to(q_2,q_4)$.

The work is organised in steps (each one has its own section below):

| Step | What it does |
|---|---|
| **1** | Original (v4) units and physics, new optimiser: population fidelity, 4th-order Yoshida kick with the exact Gaussian integral, float64 diagonalisation, CMA-ES with restarts, float64 validation in the full 128-dim space. |
| **2** | Physical units ($2\pi$ explicit, GHz/ns) and physical constraints by construction, check in the three-level transmon model. |
| **2b** | Same constraints, but the transfer must be **produced by the Floquet drive**: gate ≤ 450 ns and penalty if the transfer also happens without the pulse. |
| **3b** | Fast a-posteriori analyses of the 2b solutions: decoherence (Lindblad) and relative quasi-static noise (η scan). |
| **3** | Noise-aware optimisation (noise + decoherence in the objective), local, global, and with a constraint on the nominal F. **Interruptible**: re-running a cell resumes from the checkpoints. |

**Files.** Solutions in `results/step<X>/`, analyses in `analysis/`, PDFs in `figures/`, data in `data/`,
checkpoints in `checkpoints/` (all relative to the repository root). Every cell reads and writes through
`fv.path_for(file_name)`, which picks the folder from the file name: in a new cell just use
`open(fv.path_for('v5_step2b_1100_0011.json'))` or `plt.savefig(fv.path_for('v5_step2b_MyFigure_1100_0011.pdf'))`.

The repository ships the final solutions (`results/step2b`, `results/step3opt`) and their analyses, so the
analysis and plotting cells can be run without repeating the optimisations.

In [ ]:
%load_ext autoreload
%autoreload 2
import json, os, sys
sys.path.insert(0, os.path.abspath('..'))      # repository root (the notebook lives in notebooks/)
import numpy as np, matplotlib.pyplot as plt
from floquet import chain as fv
print('device:', fv.DEVICE)
print('available transitions:', list(fv.TRANSITIONS))
fv.pick_eigh_backend(D=64, B=32)

## Step 1 — optimisation (original units)

Put the transitions to optimise in `TO_OPTIMISE` (one, two or all three), e.g.
```python
TO_OPTIMISE = ['1001_0110']
TO_OPTIMISE = ['1100_0011', '1010_0101', '1001_0110']
```
Each transition saves its own `v5_step1_<transition>.json`.

In [ ]:
# === STEP 1: choice of the transitions ===
TO_OPTIMISE = ['1100_0011', '1010_0101', '1001_0110']

# Optional warm starts from legacy JSON solutions (folder fv.OLD_RESULTS_DIR, not shipped):
#   1100_0011, 1010_0101: found from scratch (mirror symmetry in phase A)
#   1001_0110: from scratch it stops at ~0.986; the optimum (0.993) is an extremely narrow peak -> warm start
USE_WARM  = {'1100_0011': False, '1010_0101': False, '1001_0110': True}
FTARGET   = 0.995       # stop when F_pop (float32) >= FTARGET
MAX_EVALS = 300000
MAX_TIME  = 1500        # seconds per transition (phase A)

_unknown = [nm for nm in TO_OPTIMISE if nm not in fv.TRANSITIONS]
assert not _unknown, f"unknown transitions: {_unknown} (valid: {list(fv.TRANSITIONS)})"

results_step1 = {}
for nm in TO_OPTIMISE:
    results_step1[nm] = fv.run_step1(nm, use_warm=USE_WARM.get(nm, False), ftarget=FTARGET,
                                     max_evals=MAX_EVALS, max_time=MAX_TIME)

In [ ]:
# === STEP 1 summary (reads the JSONs present, also from previous runs) ===
print(f"{'trans':10s} {'F_pop':>9s} {'F_amp':>9s} {'n':>3s} {'A':>8s} {'sigma':>7s} {'T':>8s} {'t_prot[ns]':>10s} {'errTrot':>8s} {'evals':>6s} {'t_opt[s]':>8s} {'warm':>5s}")
for nm in fv.TRANSITIONS:
    fn = f'v5_step1_{nm}.json'
    if not os.path.exists(fv.path_for(fn)):
        print(f"{nm:10s}  (no result)"); continue
    o = json.load(open(fv.path_for(fn)))
    print(f"{nm:10s} {o['fidelity']:9.5f} {o['fidelity_amp']:9.5f} {o['n_opt']:3d} {o['A_opt']:8.4f} "
          f"{o['sigma_opt']:7.3f} {o['T_opt']:8.2f} {o['protocol_time_ns']:10.1f} {o['trotter_err']:8.1e} "
          f"{o['evals']:6d} {o['opt_time_s']:8.0f} {str(o['warm_start']):>5s}")

## Step 2 — physical ranges

Physical units with explicit $2\pi$: $H = 2\pi\,[\,\sum_i -\tfrac{f_i}{2}\sigma^z_i + \sum_i g_i\,\sigma^x_i\sigma^x_{i+1} + A\,e^{-t^2/2\sigma^2}\sum_c \sigma^z_c\,]$, with f, g, A in GHz and t in ns.

The parametrisation makes the constraints true by construction:
- **qubits** $f_q\in$ `FQ`; **couplers** always *above* the neighbouring qubits by at least `DQC[0]` (dispersive regime, no strong hybridisation);
- **couplings** $g_i\in$ `G`, now optimised;
- **pulse**: $\sigma\ge$ `SIGMA[0]`; coupler excursion $2A \le$ `DF_MAX` and never below `F_FLOOR`. The coupler only moves down from its sweet spot, and even at the peak of the pulse it stays at least `KICK_DQC_MIN` above the neighbouring qubits, so there are no coupler-qubit crossings;
- **protocol duration** $\le$ `TPROT_MAX`;
- **anharmonicity**: penalty if two neighbouring qubits are on the $|11\rangle\leftrightarrow|20\rangle$ resonance, i.e. $||\Delta_{qq}|-|\alpha||<$ `ALPHA_WIN`. Every solution is then checked in the **three-level transmon model**; if the difference is > 5e-3 it is refined directly in that model (phase C).

The limits are in `fv.PHYS` and can be changed below before a run.

In [ ]:
# === STEP 2: physical limits (editable) ===
# e.g.: fv.PHYS['SIGMA'] = (1.0, 10.0)   # minimum sigma 1 ns (more aggressive choice)
for k, v in fv.PHYS.items():
    print(f"{k:10s} = {v}")

In [ ]:
# === STEP 2: choice of the transitions ===
TO_OPTIMISE_2 = ['1100_0011', '1010_0101', '1001_0110']
FTARGET_2   = 0.995
MAX_EVALS_2 = 300000
MAX_TIME_2  = 1500       # seconds per transition (phase A)

_unknown = [nm for nm in TO_OPTIMISE_2 if nm not in fv.TRANSITIONS]
assert not _unknown, f"unknown transitions: {_unknown} (valid: {list(fv.TRANSITIONS)})"

results_step2 = {}
for nm in TO_OPTIMISE_2:
    results_step2[nm] = fv.run_step2(nm, ftarget=FTARGET_2, max_evals=MAX_EVALS_2, max_time=MAX_TIME_2)

### Refinement in the transmon model (if needed)
`run_step2` already launches this refinement by itself if the result stays below 0.99 (`auto_refine=True`). This cell restarts from a saved JSON and optimises directly in the transmon model (local + random restarts). The JSON is overwritten **only if** the transmon fidelity improves.

In [ ]:
TO_REFINE = []          # e.g. ['1100_0011', '1010_0101']
for nm in TO_REFINE:
    fv.refine_step2_qutrit(nm, max_evals=40000, max_time=3000)

In [ ]:
# === STEP 2 summary with physics audit ===
rows = [('F (final model)', lambda o: f"{o['fidelity']:.5f} ({o['model_final']})"),
        ('n', lambda o: o['n_opt']),
        ('F qubit / transmon N<=4 / N<=6', lambda o: f"{o['fidelity_qubit_at_n_opt']:.4f} / {o['fidelity_qutrit_N4']:.4f} / {o['fidelity_qutrit_N6']:.4f}"),
        ('|2> leakage', lambda o: f"{o['leakage_qutrit_N4']:.1e}"),
        ('f qubit [GHz]', lambda o: np.round(o['audit']['f_qubit_GHz'], 3).tolist()),
        ('f coupler [GHz]', lambda o: np.round(o['audit']['f_coupler_GHz'], 3).tolist()),
        ('g [MHz]', lambda o: np.round(np.array(o['audit']['g_GHz']) * 1e3, 0).astype(int).tolist()),
        ('sigma [ns]', lambda o: f"{o['sigma_opt']:.2f}"),
        ('excursion 2A [GHz]', lambda o: f"{o['audit']['coupler_excursion_2A_GHz']:.3f}"),
        ('min f_c during the kick [GHz]', lambda o: f"{o['audit']['min_coupler_freq_during_kick_GHz']:.3f}"),
        ('T [ns] / period [ns]', lambda o: f"{o['T_opt']:.1f} / {o['audit']['period_ns']:.1f}"),
        ('protocol duration [ns]', lambda o: f"{o['protocol_time_ns']:.0f}"),
        ('min |Delta_qq| [GHz]', lambda o: f"{o['audit']['min_Delta_qq_GHz']:.3f}"),
        ('dist. from alpha resonance [GHz]', lambda o: f"{o['audit']['min_dist_from_alpha_resonance_GHz']:.3f}"),
        ('min Delta_qc at kick peak [GHz]', lambda o: f"{o['audit'].get('min_Delta_qc_during_kick_GHz', float('nan')):.3f}"),
        ('max g/Delta_qc', lambda o: f"{o['audit']['max_g_over_Delta_qc']:.3f}"),
        ('coupler weight in qubit modes', lambda o: f"{o['audit']['max_coupler_participation_qubit_modes']:.4f}"),
        ('Trotter error', lambda o: f"{o['trotter_err']:.1e}"),
        ('optimisation time [s]', lambda o: f"{o['opt_time_s']:.0f}")]
res = {nm: json.load(open(fv.path_for(f'v5_step2_{nm}.json'))) for nm in fv.TRANSITIONS if os.path.exists(fv.path_for(f'v5_step2_{nm}.json'))}
print(f"{'':34s}" + "".join(f"{nm:>34s}" for nm in res))
for lab, fn in rows:
    print(f"{lab:34s}" + "".join(f"{str(fn(o)):>34s}" for o in res.values()))

## Step 2b — transfer produced by the Floquet drive

The Step 2 solutions turned out to be **static** transfers: without the pulse, at a suitable time, F already reached 0.991 / 0.993 / 0.997, and the drive contributed only ~0.001. They were also slow (750–1140 ns), so decoherence cost 4–6 % and a frequency drift of a few tenths of a MHz was enough to break them.

Step 2b optimises with the **same physical constraints** as Step 2, plus three conditions:
- **gate ≤ `TPROT_MAX` = 450 ns**;
- **static penalty**: if without the pulse the fidelity exceeds `F_STATIC_MAX` = 0.3 at any time $t \le$ 1.5 µs, the candidate is penalised. The optimiser therefore has to detune the qubits and let the drive produce the transfer through a parametric resonance;
- the **first** n with F within 5·10⁻⁴ of the maximum is chosen (shortest gate for the same fidelity).

`SEARCH_MODEL = 'qutrit'` searches directly in the three-level transmon model. It is ~10 times more expensive but necessary: with large kicks the solutions found in the qubit model fail once the |2⟩ levels are included. The result goes to `v5_step2b_<name>.json`. The run is interruptible (checkpoints per phase).

In [ ]:
# === STEP 2b: parameters (editable) ===
fv.STEP2B.update(TPROT_MAX=450.0, F_STATIC_MAX=0.3, STATIC_TMAX=1500.0)
print(fv.STEP2B)

In [ ]:
# === STEP 2b: choice of the transitions ===
TO_OPTIMISE_2B = ['1100_0011', '1010_0101', '1001_0110']
SEARCH_MODEL   = 'qutrit'     # 'qutrit' (reliable, ~1 h per transition) | 'qubit' (fast, exploratory only)
MAX_TIME_2B    = 3600         # s, phase A

results_step2b = {}
for nm in TO_OPTIMISE_2B:
    # interruptible: re-running the cell resumes from the checkpoints (fresh=True to start from scratch)
    results_step2b[nm] = fv.run_step2b(nm, model_search=SEARCH_MODEL, max_time=MAX_TIME_2B,
                                       polish_evals=8000, polish_time=1200)

In [ ]:
# === STEP 2b summary: how much the drive matters ===
rows_2b = [('F with pulse (3 levels)', lambda o: f"{o['fidelity']:.5f}"),
           ('F without pulse, t <= gate', lambda o: f"{o['F_static_gate_window']:.4f}"),
           ('F without pulse, t <= 1.5 us', lambda o: f"{o['F_static_long_window']:.4f}"),
           ('Floquet contribution C', lambda o: f"{o['floquet_contribution']:.4f}"),
           ('n / gate duration [ns]', lambda o: f"{o['n_opt']} / {o['protocol_time_ns']:.0f}"),
           ('F qubit / N<=6', lambda o: f"{o['fidelity_qubit_at_n_opt']:.4f} / {o['fidelity_qutrit_N6']:.4f}"),
           ('|2> leakage', lambda o: f"{o['leakage_qutrit_N4']:.1e}"),
           ('f qubit [GHz]', lambda o: np.round(o['audit']['f_qubit_GHz'], 3).tolist()),
           ('f coupler [GHz]', lambda o: np.round(o['audit']['f_coupler_GHz'], 3).tolist()),
           ('g [MHz]', lambda o: np.round(np.array(o['audit']['g_GHz']) * 1e3, 0).astype(int).tolist()),
           ('sigma [ns] / T [ns]', lambda o: f"{o['sigma_opt']:.2f} / {o['T_opt']:.2f}"),
           ('excursion 2A [GHz]', lambda o: f"{o['audit']['coupler_excursion_2A_GHz']:.3f}"),
           ('min Delta_qc at kick peak [GHz]', lambda o: f"{o['audit']['min_Delta_qc_during_kick_GHz']:.3f}"),
           ('min |Delta_qq| [GHz]', lambda o: f"{o['audit']['min_Delta_qq_GHz']:.3f}"),
           ('dist. from alpha resonance [GHz]', lambda o: f"{o['audit']['min_dist_from_alpha_resonance_GHz']:.3f}"),
           ('max g/Delta_qc', lambda o: f"{o['audit']['max_g_over_Delta_qc']:.3f}")]
res2b = {nm: json.load(open(fv.path_for(f'v5_step2b_{nm}.json'))) for nm in fv.TRANSITIONS if os.path.exists(fv.path_for(f'v5_step2b_{nm}.json'))}
print(f"{'':38s}" + "".join(f"{nm:>34s}" for nm in res2b))
for lab, fn in rows_2b:
    print(f"{lab:38s}" + "".join(f"{str(fn(o)):>34s}" for o in res2b.values()))

## Plots of all optimised transitions
For each transition with a saved JSON the cell makes **two separate figures**: the fidelity and the (initial and final) populations as a function of n.
- `STEP_PLOT = 1`: qubit model, original units.
- `STEP_PLOT = 2`, `'2b'` or `'3opt'` (noise-optimised solutions): with `MODEL = 'auto'` the model used in the optimisation is used (usually the three-level transmon, the most realistic); `'qubit'` forces the two-level model.
- `SHOW_STATIC = True` (Step 2 onwards): adds in grey the fidelity **without the pulse** (A = 0) at the same times $t_n = n\,(T+6\sigma)$, showing how much the drive matters.
- The dotted vertical line marks n_opt. The PDFs are saved as `v5_step<N>_Fidelity_<name>.pdf` / `v5_step<N>_Populations_<name>.pdf`.

In [ ]:
# === Fidelity and population plots for all optimised transitions ===
STEP_PLOT   = '2b'      # 1 | 2 | '2b' | '3opt'
MODEL       = 'auto'    # 'auto' | 'qubit'   (only matters from Step 2 onwards)
NPLOT_ALL   = 150       # number of Floquet cycles to show
SHOW_STATIC = True      # overlay the fidelity without pulse (A = 0)
SAVE_PDF    = True

for nm in fv.TRANSITIONS:
    path = f'v5_step{STEP_PLOT}_{nm}.json'
    if not os.path.exists(fv.path_for(path)):
        print(f'{nm}: no {path}, skipping'); continue
    o = json.load(open(fv.path_for(path)))
    unit = 1.0 if STEP_PLOT == 1 else fv.UNIT_PHYS
    use_qutrit = STEP_PLOT != 1 and MODEL == 'auto' and o.get('model_final') == 'qutrit'
    if use_qutrit:
        trp = fv.qutrit(nm)
        UF  = fv.floquet_operator(trp, o['omegas_opt'], o['couplings'], o['A_opt'], o['sigma_opt'],
                                  o['T_opt'], unit, prec=fv.P64)
        i_src, i_tgt, mod_lbl = trp.src_i, trp.tgt_i, 'three-level transmon'
    else:
        UF, trp = fv.floquet_operator_full(nm, o['omegas_opt'], o['couplings'], o['A_opt'],
                                           o['sigma_opt'], o['T_opt'], unit=unit)
        i_src, i_tgt, mod_lbl = trp.src_full, trp.tgt_full, 'qubit'
    psi = np.zeros(UF.shape[0], complex); psi[i_src] = 1.0
    ns = np.arange(NPLOT_ALL + 1)
    p_src, p_tgt = np.empty(len(ns)), np.empty(len(ns))
    for n in ns:
        p_src[n], p_tgt[n] = abs(psi[i_src])**2, abs(psi[i_tgt])**2
        psi = UF @ psi
    s_lbl, t_lbl = nm.split('_')
    n_opt = o['n_opt']
    print(f"{nm} ({mod_lbl}): F(n_opt={n_opt}) = {p_tgt[n_opt]:.5f}   [JSON: {o['fidelity']:.5f}]")

    plt.figure(figsize=(11, 5))
    plt.plot(ns, p_tgt, marker='o', markersize=3, linewidth=1, linestyle='--', label='with pulse')
    if SHOW_STATIC and STEP_PLOT != 1:
        # same static Hamiltonian, no pulse, at the times t_n = n (T + 6 sigma)
        trs = trp if use_qutrit else fv.Transition(nm, reduce_parity=False)
        H0s = fv.build_H0(trs, np.asarray(o['omegas_opt'])[None], np.asarray(o['couplings']), unit)[0]
        Es, Vs = np.linalg.eigh(H0s)
        ws = Vs[i_tgt] * Vs[i_src]
        tn = ns * (o['T_opt'] + 6 * o['sigma_opt'])
        p_stat = np.abs(np.exp(-1j * np.outer(tn, Es)) @ ws) ** 2
        plt.plot(ns, p_stat, color='gray', marker='s', markersize=2, linewidth=1, label='without pulse (A = 0)')
        plt.legend(fontsize=16)
        print(f"   without pulse: max over these times = {p_stat.max():.4f}")
    plt.axvline(n_opt, color='gray', ls=':', lw=1.5)
    plt.xlabel('n (Stroboscopic Steps)', fontsize=22)
    plt.ylabel(r'Fidelity $|\langle %s|%s\rangle|^2$' % (t_lbl, s_lbl), fontsize=22)
    plt.grid(True); plt.xlim(0, NPLOT_ALL); plt.xticks(fontsize=22); plt.yticks(fontsize=22)
    plt.tight_layout()
    if SAVE_PDF: plt.savefig(fv.path_for(f'v5_step{STEP_PLOT}_Fidelity_{nm}.pdf'), bbox_inches='tight')
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.plot(ns, p_src, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Initial state population (|{s_lbl}⟩)')
    plt.plot(ns, p_tgt, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Final state population (|{t_lbl}⟩)')
    plt.axvline(n_opt, color='gray', ls=':', lw=1.5)
    plt.xlabel('n (stroboscopic steps)', fontsize=18); plt.ylabel('Population', fontsize=18)
    plt.xlim(0, NPLOT_ALL); plt.xticks(fontsize=22); plt.yticks(fontsize=22); plt.legend(fontsize=16); plt.grid(True)
    plt.tight_layout()
    if SAVE_PDF: plt.savefig(fv.path_for(f'v5_step{STEP_PLOT}_Populations_{nm}.pdf'), bbox_inches='tight')
    plt.show()

## Step 3b — a-posteriori analysis: decoherence and quasi-static noise

These analyses are fast (a few minutes per transition) and **do not enter the optimisation**.
- **Lindblad** in the three-level model with both parities ($N_{tot}\le 4$, D = 274), exact local channels (decay and dephasing). Default: qubits T1 = 50 µs, Tφ = 80 µs; couplers T1 = 10 µs, Tφ = 20 µs (order-of-magnitude, conservative values; see `fv.NOISE`). A T1 scan is also done.
- **η scan**: relative quasi-static noise, $f_i\to f_i(1+\eta Z)$, $g_i\to g_i(1+\eta Z)$, with A, T, σ noise-free (same model as the original paper). For each η: F at **fixed n** = n_opt (what is done in the lab) and the maximum of F over n (metric of the original code).

Results in `v5_step3b_<name>.json`.

In [ ]:
# === STEP 3b: a-posteriori analysis of the 2b solutions ===
TO_ANALYSE = ['1100_0011', '1010_0101', '1001_0110']
fv.ETA.update(ETA_LIST=(0.0, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3), N_SAMPLES=128)
for nm in TO_ANALYSE:
    fv.run_step3(nm, src='step2b', do_recal=False)

In [ ]:
# === Robustness curve (eta scan) and F vs T1 for all transitions ===
# Solid lines: Step 2b solutions; dashed: noise-optimised solutions (Step 3), if present.
SAVE_PDF = True
SOURCES = {'Step 2b': 'v5_step3b_', 'Step 3 (noise-opt.)': 'v5_step3opt_analysis_'}
res = {(lab, nm): json.load(open(fv.path_for(f'{pre}{nm}.json'))) for lab, pre in SOURCES.items()
       for nm in fv.TRANSITIONS if os.path.exists(fv.path_for(f'{pre}{nm}.json'))}
colors = dict(zip(fv.TRANSITIONS, plt.rcParams['axes.prop_cycle'].by_key()['color']))
style = dict(zip(SOURCES, ['-', '--']))
label = lambda lab, nm: r'$|%s\rangle\to|%s\rangle$' % tuple(nm.split('_')) + f' ({lab})'

fig, ax = plt.subplots(figsize=(8, 5.5))
for (lab, nm), r in res.items():
    L = r['eta_scan']['levels']
    ax.errorbar([l['eta'] for l in L], [l['mean'] for l in L], yerr=[l['std'] for l in L], fmt='o',
                ls=style[lab], color=colors[nm], capsize=4, label=label(lab, nm))
ax.set_xscale('symlog', linthresh=1e-6)
ax.set_xlabel(r'Relative noise level $\eta$', fontsize=18)
ax.set_ylabel(r'$\bar F(n_{opt})$', fontsize=18)
ax.tick_params(labelsize=14); ax.set_ylim(0, 1.05); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=10)
fig.tight_layout()
if SAVE_PDF: fig.savefig(fv.path_for('v5_step3_robustness_curve.pdf'), bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 5.5))
for (lab, nm), r in res.items():
    S = r['lindblad'].get('T1_scan', [])
    ax.plot([s['T1_qubit_us'] for s in S], [s['F'] for s in S], 'o', ls=style[lab], color=colors[nm],
            label=label(lab, nm))
ax.set_xscale('log'); ax.set_xlabel(r'$T_1$ qubit [$\mu$s]  ($T_\varphi$ and coupler scaled together)', fontsize=15)
ax.set_ylabel(r'$F(n_{opt})$ with decoherence', fontsize=16)
ax.tick_params(labelsize=14); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=10)
fig.tight_layout()
if SAVE_PDF: fig.savefig(fv.path_for('v5_step3_F_vs_T1.pdf'), bbox_inches='tight')
plt.show()

for (lab, nm), r in res.items():
    li = r['lindblad']
    print(f"{nm} [{lab}]: unitary F {li['F_unitary_check']:.5f} -> Lindblad {li['F_lindblad']:.5f} "
          f"(loss {li['loss_to_N<2']:.1e}, leakage {li['leak_lindblad']:.1e})")

In [ ]:
# === F(n) +- std bands for each eta (one figure per eta) ===
BAND_NAME = '1001_0110'
SOURCE    = 'v5_step3b_'          # 'v5_step3b_' (Step 2b) | 'v5_step3opt_analysis_' (Step 3, noise-opt.)
ETA_TO_SHOW = (1e-5, 1e-4, 1e-3)
SAVE_PDF = True
r = json.load(open(fv.path_for(f'{SOURCE}{BAND_NAME}.json')))
n_opt = r['n_opt']
for l in r['eta_scan']['levels']:
    if not any(np.isclose(l['eta'], e) for e in ETA_TO_SHOW):
        continue
    m, s = np.array(l['curve_mean']), np.array(l['curve_std'])
    n = np.arange(1, len(m) + 1)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(n, m, lw=2, label=r'$\bar F(n)$')
    ax.fill_between(n, m - s, m + s, alpha=0.3, label=r'$\pm\sqrt{\mathrm{Var}}$')
    ax.axvline(n_opt, color='red', ls=':', lw=1.5, label=f'$n_{{opt}}={n_opt}$')
    ax.set_xlabel('n (stroboscopic steps)', fontsize=16); ax.set_ylabel(r'$F(n)$', fontsize=16)
    ax.set_ylim(-0.05, 1.1); ax.grid(True, alpha=0.3); ax.legend(fontsize=11, loc='upper left')
    fig.tight_layout()
    eta_str = f"{l['eta']:.0e}".replace('-', 'm').replace('+', '')
    if SAVE_PDF: fig.savefig(fv.path_for(f'{SOURCE}band_{BAND_NAME}_eta{eta_str}.pdf'), bbox_inches='tight')
    plt.show()

## Step 3 — noise-aware optimisation

Starting from the Step 2b solutions, the **noise-averaged** fidelity is optimised, including decoherence:
$$\text{score} = \Big[\langle F(n)\rangle_\eta - \alpha\,\mathrm{Var}_\eta F(n)\Big]\; e^{-t_{gate}/\tau_{dec}} \;-\; \text{penalties (anharmonicity, static transfer)}$$
- $\langle\cdot\rangle_\eta$ is the average over a **fixed bank** of `K` realisations of the relative noise ($f_i, g_i \to x(1+\eta Z)$, η = `ETA_OPT`), at the same n as the nominal candidate.
- $e^{-t_{gate}/\tau_{dec}}$ accounts for decoherence; $\tau_{dec}$ is calibrated with the full Lindblad model on the starting solution (≈ 18 µs for all three 2b solutions).
- The Step 2b **static penalty** stays active, so Floquet remains essential.
- The final validation uses an **independent** bank (η scan, 128 samples) and the full Lindblad model: files `v5_step3opt_<name>.json` (solution) and `v5_step3opt_analysis_<name>.json` (analysis).

**Interruptible.** The optimisation state is saved every 30 s in `checkpoints/`. If you interrupt (■ in Jupyter, or switch off the PC), **re-running the same cell resumes where it stopped** (at most ~30 s lost). `MAX_TIME` counts the total time over all sessions. A transition already completed is skipped.
- **To start from scratch** set `RESTART_FROM_SCRATCH = True` in the run cell; the previous result is kept as `v5_step3opt_<name>_previous.json`.
- **If you change the noise parameters** (`ETA_OPT`, `K`, ...) after an interruption, the old checkpoint no longer matches: it is ignored and the run restarts from scratch automatically, so different runs are never mixed.

**Robustness target.** η = 10⁻⁴ (δf ≈ 0.5–0.65 MHz on the qubits, already 5–10 times the typical calibration accuracy) as a pessimistic but physical scenario; criterion ⟨F⟩ ≥ 0.95. η = 10⁻³ (~5 MHz) is not reachable with g ≤ 150 MHz: robustness scales as δf / J_eff and J_eff ≈ g²/Δ is at most a few tens of MHz.

In [ ]:
# === STEP 3: parameters of the noise-aware optimisation (editable) ===
fv.NOISE_OPT.update(
    ETA_OPT=1e-4,        # relative noise in the objective (~0.5-0.65 MHz on the qubits: target <F> >= 0.95)
    K=12,                # realisations per candidate
    SIGMA0_WARM=0.01,    # initial step around the 2b solution (local search: the resonances are narrow)
    ALPHA_VAR=0.1,       # weight of the variance
    USE_DEC=True,        # decoherence in the objective
    MAX_TIME=14400,      # s per transition (total over all sessions). Cost per generation with K=12:
                         # 1100 ~1 min, 1001 ~10 s -> run 1100 last, overnight (it is interruptible)
)
print(fv.NOISE_OPT)

In [ ]:
# === STEP 3: noise-aware optimisation (INTERRUPTIBLE: re-run the cell to resume) ===
TO_OPTIMISE_3 = ['1001_0110', '1010_0101', '1100_0011']   # from the fastest to the slowest
RESTART_FROM_SCRATCH = False   # False: resume from the checkpoint and skip transitions already done
                               # True : ignore checkpoints and previous results and start again
                               #        (the old result is moved to v5_step3opt_<name>_previous.json)
# A result obtained with noise parameters different from the current ones (ETA_OPT, K, ...) is redone
# automatically; the old one is kept as v5_step3opt_<name>_eta<old ETA>.json (with its analysis).
KEYS = ('ETA_OPT', 'K', 'BANK_SEED', 'ALPHA_VAR', 'USE_DEC')
for nm in TO_OPTIMISE_3:
    fn = f'v5_step3opt_{nm}.json'
    if os.path.exists(fv.path_for(fn)):
        old = json.load(open(fv.path_for(fn))).get('NOISE_OPT', {})
        same = all(old.get(k) == fv.NOISE_OPT[k] for k in KEYS)
        if same and not RESTART_FROM_SCRATCH:
            print(f"{nm}: already done with these parameters, skipping (RESTART_FROM_SCRATCH = True to redo it)"); continue
        suff = '_previous' if same else f"_eta{old.get('ETA_OPT', 0):.0e}"
        for pre in ('v5_step3opt_', 'v5_step3opt_analysis_'):
            if os.path.exists(fv.path_for(f'{pre}{nm}.json')):
                os.replace(fv.path_for(f'{pre}{nm}.json'), fv.path_for(f'{pre}{nm}{suff}.json'))
        print(f"{nm}: previous result moved to v5_step3opt_{nm}{suff}.json")
    fv.run_step3_opt(nm, fresh=RESTART_FROM_SCRATCH)

In [ ]:
# === STEP 3 summary: before (Step 2b) and after the noise-aware optimisation ===
def summary_rows(nm):
    p = f'v5_step3opt_{nm}.json'
    if not os.path.exists(fv.path_for(p)):
        return None
    o, a = json.load(open(fv.path_for(p))), json.load(open(fv.path_for(f'v5_step3opt_analysis_{nm}.json')))
    b = json.load(open(fv.path_for(f'v5_step3b_{nm}.json'))) if os.path.exists(fv.path_for(f'v5_step3b_{nm}.json')) else None
    eta = o['NOISE_OPT']['ETA_OPT']
    at = lambda r: next((l for l in r['eta_scan']['levels'] if np.isclose(l['eta'], eta)), None)
    s0, s1 = o['objective_start'], o['objective_final']
    return [('objective score', s0['score'], s1['score']),
            ('nominal F', s0['F'], o['fidelity']),
            ('F with decoherence (Lindblad)', b['lindblad']['F_lindblad'] if b else np.nan, a['lindblad']['F_lindblad']),
            (f'<F> eta={eta:.0e} (indep. bank)', at(b)['mean'] if b and at(b) else np.nan, at(a)['mean'] if at(a) else np.nan),
            (f'std F eta={eta:.0e}', at(b)['std'] if b and at(b) else np.nan, at(a)['std'] if at(a) else np.nan),
            ('gate duration [ns]', s0['gate'], o['protocol_time_ns']),
            ('F without pulse', s0['Fstat'], o['F_static_long_window'])]
for nm in fv.TRANSITIONS:
    R = summary_rows(nm)
    if R is None:
        print(f'{nm}: not optimised yet'); continue
    print(f'\n{nm}' + ' ' * 24 + f"{'Step 2b':>12s}{'Step 3':>12s}")
    for lab, x0, x1 in R:
        print(f'  {lab:34s}{x0:12.5f}{x1:12.5f}')

### Global Step 3 — for transitions where the local search is not enough (1100_0011)

The local search cannot leave a narrow resonance. On 1100_0011 the 2b solution has all 4 qubits at ~6.49 GHz and g ≈ 40–90 MHz, hence a small $J_\text{eff}$; in 4 hours the local search found no better candidate. Robustness scales as $\delta f / J_\text{eff}$, so a different **geometry** is needed (lower qubits, larger g), searched with the noise already in the objective.

- **Phase G** (global, ~6 h). Mirror constraints and K=4 realisations. Candidates with nominal F < 0.5 are not simulated with noise, so they are cheap. Warm starts from the solutions of the other transitions plus random IPOP restarts. The noise is applied to every site independently, so it breaks the symmetry as in the lab.
- **Phase L** (local, ~2 h). Refinement in the full space with the standard objective (K=12), local restarts only.
- **Outcome.** The result replaces `v5_step3opt_<name>.json` only if the score improves (the old one becomes `*_local.json`); otherwise it is saved as `v5_step3opt_<name>_global.json`. Interruptible like Step 3.

For 1100_0011 the shipped files are: `v5_step3opt_1100_0011.json` = global result (n = 3, F = 0.984, more robust, 130 ns) and `v5_step3opt_1100_0011_local.json` = local result (n = 4, F = 0.9956, 253 ns, identical to the 2b solution).

In [ ]:
# === GLOBAL STEP 3 (INTERRUPTIBLE: re-run the cell to resume) ===
TO_SEARCH_GLOBALLY = ['1100_0011']
fv.NOISE_GLOBAL.update(
    TIME_G=6 * 3600.0,     # s, global phase
    TIME_L=2 * 3600.0,     # s, local refinement
)
RESTART_FROM_SCRATCH_G = False
for nm in TO_SEARCH_GLOBALLY:
    fv.run_step3_global(nm, fresh=RESTART_FROM_SCRATCH_G)

### Step 3 — refinement with a constraint on the nominal F
Noise-optimised solutions can give up nominal F in exchange for robustness (1100 global: F = 0.984). This cell restarts from `v5_step3opt_<name>.json` with the same objective plus the soft constraint score −= W·max(0, F_MIN − F), with local restarts only. The result does **not** replace the 3opt solution: it goes to `v5_step3opt_<name>_fmin.json`, with the analysis in `v5_step3opt_analysis_<name>_fmin.json`. The next cell compares the two. (For 1100 and 1010 this refinement found no better point: the two optima are separated.)

In [ ]:
# === STEP 3: REFINEMENT WITH NOMINAL F >= F_MIN (INTERRUPTIBLE: re-run the cell to resume) ===
TO_REFINE_F = ['1100_0011']
fv.NOISE_FMIN.update(
    F_MIN=0.9905,          # threshold on the nominal F (0.99 + margin)
    W=10.0,                # weight of the constraint
    MAX_TIME=2 * 3600.0,   # s
)
RESTART_FROM_SCRATCH_F = False
for nm in TO_REFINE_F:
    fv.run_step3_fmin(nm, fresh=RESTART_FROM_SCRATCH_F)

In [ ]:
# === Comparison: 3opt solution vs refinement with F_MIN ===
for nm in ['1100_0011', '1010_0101', '1001_0110']:
    rows = []
    for lab, suf in (('3opt', ''), ('3opt F_MIN', '_fmin')):
        fs, fa = fv.path_for(f'v5_step3opt_{nm}{suf}.json'), fv.path_for(f'v5_step3opt_analysis_{nm}{suf}.json')
        if not (os.path.exists(fs) and os.path.exists(fa)):
            continue
        o, a = json.load(open(fs)), json.load(open(fa))
        L = {round(l['eta'], 7): l['mean'] for l in a['eta_scan']['levels']}
        rows.append((lab, o['fidelity'], a['lindblad']['F_lindblad'], L.get(3e-5), L.get(1e-4), L.get(3e-4),
                     o['protocol_time_ns'], o['floquet_contribution'], o['objective_final']['score']))
    if len(rows) < 2:
        continue
    print(f"\n{nm}\n  {'':12s}{'F':>8s}{'Lindblad':>10s}{'<F>3e-5':>9s}{'<F>1e-4':>9s}{'<F>3e-4':>9s}{'gate':>7s}{'C':>7s}{'score':>8s}")
    for r in rows:
        print(f"  {r[0]:12s}{r[1]:8.4f}{r[2]:10.4f}{r[3]:9.4f}{r[4]:9.4f}{r[5]:9.4f}{r[6]:7.0f}{r[7]:7.3f}{r[8]:8.4f}")
print("\nNote: the score of the F_MIN version includes the penalty on the constraint (if F < F_MIN).")

## Noise analysis: robustness curve (as in the paper)

**Relative** quasi-static noise: for each sample $f_i \to f_i(1+\eta Z_i)$ (qubits and couplers) and $g_i \to g_i(1+\eta Z_i)$, with $Z \sim \mathcal N(0,1)$, while $A$, $T$ and $\sigma$ stay noise-free. The computation uses the three-level transmon model and the same samples $Z$ for all $\eta$, so the curves are smooth.

- **fixed n** (filled markers): $\bar F$ at the fixed number of kicks $n_{opt}$. This is what is done in the lab, since $n$ cannot be chosen sample by sample.
- **max over n** (dashed, optional): $\bar F_{\max}$, the maximum over $n$ for each sample, as in the figure of the original paper. More optimistic.

For reference, $\eta = 10^{-4}$ corresponds to $\delta f_q \approx 0.5$–$0.65$ MHz. The results go to `v5_eta_<source>_<name>.json` and are reloaded if the parameters do not change (`RECOMPUTE = True` to redo the computation). The solution files are `v5_step<SOURCE>_<name><SUFFIX>.json`: e.g. `SOURCE = '3opt'`, `SUFFIX = '_local'` for the n = 4 solution of 1100_0011.

In [ ]:
# === Noise analysis: F as a function of eta (robustness curve) ===
import time
NAMES     = ['1100_0011', '1010_0101', '1001_0110']
SOURCE    = '2b'          # '2b' | '3opt' | 2   (solution to analyse: v5_step<SOURCE>_<name><SUFFIX>.json)
SUFFIX    = ''            # '' | '_local' | '_fmin'
ETA_LIST  = [0, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3]
N_SAMPLES = 128           # noise realisations per eta
SHOW_MAX_N   = True       # adds F_max over n (metric of the original paper), dashed
ONE_FIGURE   = False      # True: all transitions in the same plot
MHZ_AXIS     = True       # top axis with the absolute error on the qubit frequencies, delta f_q [MHz]
ETA_REALISTIC = 1e-4      # shaded region (0 = none): delta f_q <~ 0.5 MHz
RECOMPUTE    = False
SAVE_PDF     = True

def robustness_scan(nm):
    sol = f'v5_step{SOURCE}_{nm}{SUFFIX}.json'
    if not os.path.exists(fv.path_for(sol)):
        print(f'{nm}: {sol} missing, skipping'); return None
    o = json.load(open(fv.path_for(sol)))
    cache = f'v5_eta_{SOURCE}_{nm}{SUFFIX}.json'
    par = dict(sol=sol, params=[o['omegas_opt'], o['couplings'], o['A_opt'], o['sigma_opt'], o['T_opt'], o['n_opt']],
               eta=list(map(float, ETA_LIST)), n=N_SAMPLES)
    if not RECOMPUTE and os.path.exists(fv.path_for(cache)):
        c = json.load(open(fv.path_for(cache)))
        if c.get('par') == json.loads(json.dumps(par)):
            print(f'{nm}: read from {cache}'); return c
    print(f'{nm}: computing ({len(ETA_LIST)} eta x {N_SAMPLES} samples)...')
    t0 = time.time()
    r = fv.eta_scan(nm, o, eta_list=ETA_LIST, n_samples=N_SAMPLES)
    c = dict(par=par, scan=r, F_nominal=o['fidelity'], n_opt=o['n_opt'], time_s=time.time() - t0)
    json.dump(c, open(fv.path_for(cache), 'w'), indent=1)
    print(f'   done in {c["time_s"]:.0f} s -> {cache}')
    return c

res = {nm: robustness_scan(nm) for nm in NAMES}
res = {nm: c for nm, c in res.items() if c}
colors = dict(zip(fv.TRANSITIONS, plt.rcParams['axes.prop_cycle'].by_key()['color']))
ket = lambda nm: r'$|%s\rangle\to|%s\rangle$' % tuple(nm.split('_'))

def draw(ax, nm, c, multi):
    L = c['scan']['levels']
    eta = np.array([l['eta'] for l in L])
    col = colors[nm] if multi else 'steelblue'
    ax.errorbar(eta, [l['mean'] for l in L], yerr=[l['std'] for l in L], fmt='o-', capsize=5, color=col,
                label=(ket(nm) + ', ' if multi else '') + r'$\bar F(n_{opt}=%d)$' % c['n_opt'])
    if SHOW_MAX_N:
        ax.errorbar(eta, [l['mean_maxn'] for l in L], yerr=[l['std_maxn'] for l in L], fmt='s--', capsize=3,
                    color=col, alpha=0.55, ms=4, label=(ket(nm) + ', ' if multi else '') + r'$\bar F_{\max}$ over $n$')
    if not multi:
        ax.axhline(L[0]['mean'], color='gray', ls='--', lw=1, label=f"Ideal fidelity = {L[0]['mean']:.4f}")

def finish(ax, title, cs):
    ax.set_xscale('symlog', linthresh=1e-6)
    ax.set_xlim(-2e-7, max(ETA_LIST) * 1.6)
    if ETA_REALISTIC:
        ax.axvspan(0, ETA_REALISTIC, color='tab:green', alpha=0.08, lw=0,
                   label=r'realistic static error ($\eta \leq 10^{%d}$)' % round(np.log10(ETA_REALISTIC)))
    if MHZ_AXIS:
        # delta f_q = eta * <f_q> (mean over the qubits); with several transitions the mean value is used
        k = np.mean([l['df_qubit_MHz'] / l['eta'] for c in cs for l in c['scan']['levels'] if l['eta'] > 0])
        top = ax.secondary_xaxis('top', functions=(lambda x: x * k, lambda y: y / k))
        top.set_xscale('symlog', linthresh=1e-6 * k)
        tk = [t for t in (0.01, 0.1, 1, 10) if t <= k * max(ETA_LIST) * 1.6]
        top.set_xticks(tk, [f'{t:g}' for t in tk]); top.minorticks_off()
        top.set_xlabel(r'$\delta f_q$ [MHz]' + ('' if len(cs) == 1 else ' (mean over transitions)'), fontsize=14)
        top.tick_params(labelsize=12)
    ax.set_xlabel(r'Relative noise level  $\eta$', fontsize=20)
    ax.set_ylabel(r'$\bar F$', fontsize=20)
    ax.tick_params(axis='both', which='major', labelsize=15)
    ax.set_title(title, fontsize=19)
    ax.set_ylim(0, 1.05); ax.grid(True, which='both', alpha=0.3); ax.legend(fontsize=11, loc='lower left')

if ONE_FIGURE:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    for nm, c in res.items():
        draw(ax, nm, c, True)
    finish(ax, 'Robustness curve', list(res.values())); fig.tight_layout()
    if SAVE_PDF: fig.savefig(fv.path_for(f'v5_robustness_curve_{SOURCE}{SUFFIX}_all.pdf'), bbox_inches='tight')
    plt.show()
else:
    for nm, c in res.items():
        fig, ax = plt.subplots(figsize=(7, 5))
        draw(ax, nm, c, False)
        finish(ax, 'Robustness curve ' + ket(nm), [c]); fig.tight_layout()
        if SAVE_PDF: fig.savefig(fv.path_for(f'v5_robustness_curve_{SOURCE}_{nm}{SUFFIX}.pdf'), bbox_inches='tight')
        plt.show()

for nm, c in res.items():
    print(f"\n{nm}  (step {SOURCE}{SUFFIX}, nominal F {c['F_nominal']:.5f}, n_opt = {c['n_opt']})")
    print(f"  {'eta':>8} {'df_q [MHz]':>11} {'<F> fixed n':>12} {'std':>8} {'p20':>8} {'min':>8} {'<F_max> over n':>15}")
    for l in c['scan']['levels']:
        print(f"  {l['eta']:8.0e} {l['df_qubit_MHz']:11.3f} {l['mean']:12.5f} {l['std']:8.5f} {l['p20']:8.5f} "
              f"{l['min']:8.5f} {l['mean_maxn']:15.5f}")

## Analysis of one solution

Choose `NAME` (transition), `STEP` (1 = original units, 2, '2b' or '3opt') and optionally `SUFFIX` (e.g. `'_local'`). The `v5_step3_*` and `v5_step3b_*` files are noise analyses, not solutions, and cannot be used here. With `MODEL = 'auto'` the model used in the optimisation is used: three-level transmon (parity sector, ~190 states) for 2b and 3opt, qubit (128 states) otherwise; `'qubit'` forces the two-level model.

The next cell builds the exact Floquet operator (float64), computes the stroboscopic evolution up to `NPLOT` steps and makes the plots. It defines `Floquet`, `T_opt`, `fids`, `ket_source`, `ket_target`, used by the resolvent and FFT cells below. The Floquet period used in those cells is the **full** period $T + 6\sigma$ (pulse window + free evolution).

In [ ]:
# === Stroboscopic evolution, plots and data export ===
NAME    = '1100_0011'
STEP    = '2b'         # 1 | 2 | '2b' | '3opt'   (NOT 3 / '3b': those are analysis files, not solutions)
SUFFIX  = ''           # '' | '_local' | '_fmin'
MODEL   = 'auto'       # 'auto' (model of the optimisation) | 'qubit'
NPLOT   = 10000        # steps of the evolution (many points are needed for the FFT)
XMAX    = 150          # zoom of the plots
SHOW_STATIC = True     # overlay the fidelity without pulse (A = 0) at the same times t_n = n (T + 6 sigma)

path = f'v5_step{STEP}_{NAME}{SUFFIX}.json'
if not os.path.exists(fv.path_for(path)):
    raise FileNotFoundError(f"{path} does not exist: choose STEP among 1, 2, '2b', '3opt' with a solution already computed")
o = json.load(open(fv.path_for(path)))
if 'T_opt' not in o:
    raise ValueError(f"{path} is an analysis file, not a solution: use STEP = '2b' or '3opt'")
unit  = 1.0 if STEP == 1 else fv.UNIT_PHYS
T_opt = o['T_opt']; n_best = o['n_opt']
if STEP != 1 and MODEL == 'auto' and o.get('model_final') == 'qutrit':
    trq = fv.qutrit(NAME)
    Floquet = np.asarray(fv.floquet_operator(trq, o['omegas_opt'], o['couplings'], o['A_opt'], o['sigma_opt'],
                                             T_opt, unit, prec=fv.P64))
    i_src, i_tgt, mod_lbl = trq.src_i, trq.tgt_i, 'three-level transmon'
    trs = trq
else:
    Floquet, tr = fv.floquet_operator_full(NAME, o['omegas_opt'], o['couplings'], o['A_opt'],
                                           o['sigma_opt'], T_opt, unit=unit)
    i_src, i_tgt, mod_lbl = tr.src_full, tr.tgt_full, 'qubit'
    trs = fv.Transition(NAME, reduce_parity=False)
dim = Floquet.shape[0]
ket_source = np.zeros(dim, complex); ket_source[i_src] = 1.0
ket_target = np.zeros(dim, complex); ket_target[i_tgt] = 1.0
src_lbl, tgt_lbl = NAME.split('_')
tag = f'{STEP}{SUFFIX}'
print(f"{NAME} [step {tag}, {mod_lbl}, dim {dim}]: F={o['fidelity']:.6f} at n={n_best} | "
      f"A={o['A_opt']:.4f}, sigma={o['sigma_opt']:.4f}, T={T_opt:.4f}")

ns = np.arange(NPLOT + 1)
fids, p_src = np.empty(NPLOT + 1), np.empty(NPLOT + 1)
psi = ket_source.copy()
for n in ns:
    fids[n]  = abs(np.vdot(ket_target, psi))**2
    p_src[n] = abs(np.vdot(ket_source, psi))**2
    psi = Floquet @ psi
p_tgt = fids

out_txt = f'v5_step{STEP}_evolution_{NAME}{SUFFIX}.txt'
np.savetxt(fv.path_for(out_txt), np.column_stack([ns, fids, p_src, p_tgt]), fmt=['%d', '%.8f', '%.8f', '%.8f'],
           header=f"7-site evolution, Gaussian pulse, {NAME}, step {tag}\n"
                  f"omegas={o['omegas_opt']}\nA={o['A_opt']:.6f} sigma={o['sigma_opt']:.6f} T={T_opt:.4f} n_best={n_best}\n"
                  "n fidelity initial_population final_population")
print('Saved', out_txt)

plt.figure(figsize=(11, 5))
plt.plot(ns, fids, marker='o', markersize=3, linewidth=1, linestyle='--', label='with pulse')
if SHOW_STATIC:
    # same static Hamiltonian, no pulse, at the times t_n = n (T + 6 sigma)
    H0s = fv.build_H0(trs, np.asarray(o['omegas_opt'])[None], np.asarray(o['couplings']), unit)[0]
    H0s = H0s.cpu().numpy() if hasattr(H0s, 'cpu') else np.asarray(H0s)
    Es, Vs = np.linalg.eigh(H0s)
    ws = Vs[i_tgt] * Vs[i_src]
    n_st = ns[:XMAX + 1]
    p_stat = np.abs(np.exp(-1j * np.outer(n_st * (T_opt + 2 * fv.WINDOW * o['sigma_opt']), Es)) @ ws) ** 2
    plt.plot(n_st, p_stat, color='gray', marker='s', markersize=2, linewidth=1, label='without pulse (A = 0)')
    plt.legend(fontsize=16)
    print(f"   without pulse: max over these times = {p_stat.max():.4f}")
plt.axvline(n_best, color='gray', ls=':', lw=1.5)
plt.xlabel('n (Stroboscopic Steps)', fontsize=22)
plt.ylabel(r'Fidelity $|\langle %s|%s\rangle|^2$' % (tgt_lbl, src_lbl), fontsize=22)
plt.grid(True); plt.xlim(0, XMAX); plt.xticks(fontsize=22); plt.yticks(fontsize=22)
plt.tight_layout(); plt.savefig(fv.path_for(f'v5_step{STEP}_Fidelity_{NAME}{SUFFIX}.pdf'), bbox_inches='tight'); plt.show()

plt.figure(figsize=(12, 6))
plt.plot(ns, p_src, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Initial state population (|{src_lbl}⟩)')
plt.plot(ns, p_tgt, marker='o', markersize=3, linewidth=1, linestyle='--', label=f'Final state population (|{tgt_lbl}⟩)')
plt.xlabel('n (stroboscopic steps)', fontsize=18); plt.ylabel('Population', fontsize=18)
plt.xlim(0, XMAX); plt.xticks(fontsize=22); plt.yticks(fontsize=22); plt.legend(fontsize=16); plt.grid(True)
plt.tight_layout(); plt.savefig(fv.path_for(f'v5_step{STEP}_Populations_{NAME}{SUFFIX}.pdf'), bbox_inches='tight'); plt.show()

In [ ]:
# === Resolvent and quasi-energies (EXACT Floquet operator, finite-width kick) ===
from scipy.linalg import eig

T = T_opt + 2 * fv.WINDOW * o['sigma_opt']   # FULL Floquet period: pulse window (6 sigma) + free evolution T_opt
UF = Floquet
print(f"Floquet period T = T_opt + 6 sigma = {T:.4f} ns")

# eigenvalues and eigenvectors of UF
eigvals, eigvecs = eig(UF)
for j in range(dim):
    eigvecs[:, j] /= np.linalg.norm(eigvecs[:, j])

# quasi-energies
thetas = -np.angle(eigvals)
thetas = (thetas + np.pi) % (2 * np.pi) - np.pi
eps = thetas / T

# residues r_alpha
istate, fstate = ket_source, ket_target
r = np.array([np.vdot(fstate, eigvecs[:, j]) * np.vdot(eigvecs[:, j], istate) for j in range(dim)])

# resolvent
def Gofz(z):
    return np.sum(r / (1 - z * eigvals))

wmin, wmax, npts = -np.pi / T, np.pi / T, 100000
wgrid = np.linspace(wmin, wmax, npts)
Gvals = np.array([Gofz(np.exp(-1j * w * T)) for w in wgrid])
Gmag  = np.abs(Gvals)

plt.figure(figsize=(9, 6))
plt.plot(wgrid, Gmag, 'b-', lw=2, label=r'$|G(e^{-i\omega T})|$')
for i, e in enumerate(eps):
    plt.axvline(e, color='crimson', linestyle='--', lw=1, alpha=0.5,
                label=r'$\epsilon_\alpha$' if i == 0 else None)
plt.xlabel(r'$\omega$', fontsize=12); plt.ylabel(r'$|G(e^{-i\omega T})|$', fontsize=12)
plt.title(f'Floquet channels and quasi-energy poles (exact Gaussian kick) — {NAME}', fontsize=13)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(fv.path_for(f'v5_step{STEP}_Resolvent_{NAME}{SUFFIX}.pdf'), bbox_inches='tight'); plt.show()

np.savetxt(fv.path_for(f'v5_step{STEP}_floquet_eps_{NAME}{SUFFIX}.dat'), eps)
np.savetxt(fv.path_for(f'v5_step{STEP}_floquet_eps_{NAME}{SUFFIX}_sorted.dat'), np.sort(eps))

unitarity_error = np.linalg.norm(UF.conj().T @ UF - np.eye(dim), np.inf)
print("Unitarity error (exact UF):", unitarity_error)
print("Dominant residues:", np.sort(np.abs(r))[::-1][:10])

In [ ]:
# === Simple FFT of the fidelity (omega basis) ===
fidelity_data = np.array(fids)
N_data = len(fidelity_data)

fft_result = np.fft.fft(fidelity_data)
amplitude_spectrum = np.abs(fft_result) / N_data

freq_axis_nu = np.fft.fftfreq(N_data, d=1.0)
fft_shifted_nu = np.fft.fftshift(amplitude_spectrum)
freq_shifted_nu = np.fft.fftshift(freq_axis_nu)
omega_fft_axis = (2 * np.pi / T) * freq_shifted_nu   # T = full period (resolvent cell)

plt.figure(figsize=(10, 5))
plt.plot(omega_fft_axis, fft_shifted_nu, 'b-', lw=2)

fft_shifted_display = fft_shifted_nu.copy()
fft_shifted_display[np.argmin(np.abs(omega_fft_axis))] = 0.0
peak_indices = np.argsort(fft_shifted_display)[::-1][:3]
peak_omegas, peak_amps = omega_fft_axis[peak_indices], fft_shifted_nu[peak_indices]
for omega, amp in zip(peak_omegas, peak_amps):
    if amp > 0.01:
        plt.scatter(omega, amp, color='red', marker='o')
        plt.annotate(f'{amp:.3f}\n({omega:.4f})', (omega, amp),
                     textcoords="offset points", xytext=(5, -10), ha='center')

plt.title(r'Floquet frequency spectrum of the fidelity ($\omega$ basis)')
plt.xlabel(r'Quasi-energy $\omega$', fontsize=12); plt.ylabel('Amplitude (Fourier spectrum)', fontsize=12)
plt.grid(True); plt.xlim(-np.pi / T, np.pi / T); plt.tight_layout(); plt.show()

print("\n--- Spectrum analysis ---")
for omega, amp in sorted(zip(peak_omegas, peak_amps), key=lambda x: x[1], reverse=True):
    if amp > 0.01:
        print(f"  omega: {omega:.5f}, amplitude: {amp:.5f}")

In [ ]:
# === Windowed FFT + comparison with the quasi-energies eps ===
from scipy.signal import find_peaks, windows

signal = np.array(fids)
N = signal.size
T = T_opt + 2 * fv.WINDOW * o['sigma_opt']   # full Floquet period (6 sigma pulse window + free evolution)

sig_win = signal * windows.hann(N)
pad_factor = 8
Npad = int(N * pad_factor)
sig_pad = np.pad(sig_win, (0, Npad - N), mode='constant')

S = np.fft.fft(sig_pad)
omega_axis = 2 * np.pi * np.fft.fftfreq(Npad, d=T)
S_mag = np.abs(S)
omega_axis_shifted = np.fft.fftshift(omega_axis)
S_mag_shifted = np.fft.fftshift(S_mag)

half = Npad // 2
omega_pos, Spos = omega_axis_shifted[half:], S_mag_shifted[half:]
peaks, props = find_peaks(Spos, height=np.max(Spos) * 0.05, distance=3)
peak_omegas, peak_heights = omega_pos[peaks], props['peak_heights']

eps_wrapped = ((eps + np.pi / T) % (2 * np.pi / T)) - np.pi / T
matches = []
for om in peak_omegas:
    i_c = np.argmin(np.abs(eps_wrapped - om))
    matches.append((om, eps_wrapped[i_c], eps[i_c], np.abs(eps_wrapped[i_c] - om)))

print("Angular resolution Δω =", 2 * np.pi / (Npad * T))
print("Peaks found (omega, matched_eps_wrapped, original_eps, distance):")
for m in matches:
    print(f"{m[0]:.6e}   {m[1]:.6e}   {m[2]:.6e}   diff = {m[3]:.3e}")

plt.figure(figsize=(9, 4))
plt.plot(omega_axis_shifted, S_mag_shifted, label='|FFT(signal)| (shifted)')
for e in eps_wrapped:
    plt.axvline(e, color='crimson', linestyle='--', alpha=0.6)
plt.xlim(-np.pi / T, np.pi / T)
plt.xlabel(r'$\omega$ (rad/ns)'); plt.ylabel('FFT amplitude')
plt.title('Fidelity spectrum (FFT) and quasi-energy lines')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# === Delta-epsilon transitions weighted by the residues, overlaid on the spectrum ===
c = np.array([np.vdot(eigvecs[:, a], ket_source) for a in range(dim)])
d = np.array([np.vdot(ket_target, eigvecs[:, a]) for a in range(dim)])

DE_mat = eps[:, None] - eps[None, :]
DE_mat = ((DE_mat + np.pi / T) % (2 * np.pi / T)) - np.pi / T
WE_mat = np.abs((c[:, None] * np.conj(c)[None, :]) * (d[None, :] * np.conj(d)[:, None]))
DE, WE = DE_mat.ravel(), WE_mat.ravel()
pairs = [(a, b) for a in range(dim) for b in range(dim)]

topK = 50
top_idx = np.argsort(WE)[::-1][:topK]
print("Top transitions (a,b) with wrapped frequency and weight:")
for ii in top_idx:
    a, b = pairs[ii]
    print(f"({a},{b})  Δε_wrapped = {DE[ii]:.6e}   weight = {WE[ii]:.3e}")

plt.figure(figsize=(11, 5))
for ii in top_idx:
    plt.axvline(DE[ii], color='red', linestyle='--', linewidth=1, alpha=0.6)
plt.plot(omega_axis_shifted, S_mag_shifted, lw=1, color='tab:blue')
plt.xlim(-np.pi / T, np.pi / T)
plt.xlabel(r'$\omega$ (rad/ns)', fontsize=20); plt.ylabel('FFT Amplitude', fontsize=20)
plt.grid(alpha=0.3)
plt.gca().ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
plt.xticks(fontsize=18); plt.yticks(fontsize=18); plt.tight_layout()
plt.savefig(fv.path_for(f'v5_step{STEP}_comparison_{NAME}{SUFFIX}.pdf'), bbox_inches='tight'); plt.show()